In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Get your key from environment
google_key = os.getenv("GOOGLE_API_KEY")
print(f"Key exists: {bool(google_key)}")

# Create the model
model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",  # Use flash for free tier
    google_api_key=google_key
)

Key exists: True


In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model = model ,
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt="You are a travel agent. No follow up questions."
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on March 31st")]},
    config
    )

Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='2c01a777-930f-4e75-a7f0-bc64d95b9c5f'),
              AIMessage(content=[{'type': 'text', 'text': "I can't search for direct flights only, but I can search for flights from San Francisco to Tokyo on March 31st. Would you like me to do that?", 'extras': {'signature': 'CtgGAb4+9vuqXCaYak0gURTW43YnqxzMCE2X3DnN3/y/GJ0LlZd9oHL2LSjnOeuS1R3tcMAZyOpcEynLeVYTw3BtkHbmjF1Ahk9/w8fubUsNzddrGN76Pv4URyCliBkztKuINHn2l8yL7wAF9Jeh5Ro1LXFGxmyEQmdKKF7P/ShGKWLFg5+7Gn/APJnJErFD2usQ7v3F27+Ni+WlSoTgGZjg0MYy+KVfJqhM7jOwJT1nFxqYHVsTFhmoq5sqv7gQ5+sElmO7QwbW/VydNotu/92Pa0PIv5yag6QqFcgKCFw9VdcUPWkRtbTuFW/2H5R4SYGTi8VNQokT0QAaqljJpd0nFLhtOH2QR2tGo+KtBIYNo6mj7krp90VuKJbJDKPt8iUpxlo0p5pVR9RkUx8Io1QNzb0CtDpHfCpUhbEBsQG8EGJkB0lpHRjAahNih7QRiiNOtGw26q82vdmTO6WMcH9ZRAGShwvsuNkkeg58wPytEa8zQq/VMgGFI6lxCByZnaAeVRG16m5lhPkvdVXVVUWEmnOkbUEj/9u9QEOP9FFkyopzROfdecb/wuTffwK/lK0Lq

In [8]:
print(response["messages"][-1].content)

[{'type': 'text', 'text': "I can't search for direct flights only, but I can search for flights from San Francisco to Tokyo on March 31st. Would you like me to do that?", 'extras': {'signature': 'CtgGAb4+9vuqXCaYak0gURTW43YnqxzMCE2X3DnN3/y/GJ0LlZd9oHL2LSjnOeuS1R3tcMAZyOpcEynLeVYTw3BtkHbmjF1Ahk9/w8fubUsNzddrGN76Pv4URyCliBkztKuINHn2l8yL7wAF9Jeh5Ro1LXFGxmyEQmdKKF7P/ShGKWLFg5+7Gn/APJnJErFD2usQ7v3F27+Ni+WlSoTgGZjg0MYy+KVfJqhM7jOwJT1nFxqYHVsTFhmoq5sqv7gQ5+sElmO7QwbW/VydNotu/92Pa0PIv5yag6QqFcgKCFw9VdcUPWkRtbTuFW/2H5R4SYGTi8VNQokT0QAaqljJpd0nFLhtOH2QR2tGo+KtBIYNo6mj7krp90VuKJbJDKPt8iUpxlo0p5pVR9RkUx8Io1QNzb0CtDpHfCpUhbEBsQG8EGJkB0lpHRjAahNih7QRiiNOtGw26q82vdmTO6WMcH9ZRAGShwvsuNkkeg58wPytEa8zQq/VMgGFI6lxCByZnaAeVRG16m5lhPkvdVXVVUWEmnOkbUEj/9u9QEOP9FFkyopzROfdecb/wuTffwK/lK0LqPcrHbPih+S96As+q4O6wUk/RD0zjnAVsyx89BvBiHlpxVX0vZP1bpTRe5Jak9Iorht6RQISaQOkEv5PjjtJ+zdL9LSiBMkeLd0FztNuJxIJt5RdFkVdQsO37NMQ8LpdCO0dGYPC+J81mXd7JHI1QehnlkQ4VzPnEl8BID5tKdmXA0zS9/bv/1Dw5tEUpmfs0haMZlBSSws3cjAilLF6KK/Ozzk1hqcb

In [9]:
def clean_response(response):
    """Extract clean text from agent response"""
    last_msg = response['messages'][-1]
    
    if isinstance(last_msg.content, list):
        # Handle Gemini's list format
        for item in last_msg.content:
            if isinstance(item, dict) and item.get('type') == 'text':
                return item['text']
    else:
        return last_msg.content
    
    return "No text content found"

print(clean_response(response))

I can't search for direct flights only, but I can search for flights from San Francisco to Tokyo on March 31st. Would you like me to do that?


In [ ]:
## https://mcp.so/servers
## glama.ai\mcp\servers